<a href="https://colab.research.google.com/github/Lattemelia/Portfolio/blob/main/%E3%80%900624%EF%BD%9E%E7%8F%BE%E5%9C%A8%E4%BD%9C%E6%88%90%E4%B8%AD%E3%80%91/%E4%B8%AD%E5%8F%A4%E8%BB%8A%E3%81%AE%E4%BE%A1%E6%A0%BC%E4%BA%88%E6%B8%AC(KaggleConpe).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 新しいセクション

In [1]:
#デ－タの読み込み
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
#ツールの読み込み
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
#データをデータフレームに落とし込み
PATH = r"/content/drive/MyDrive/Colab Notebooks"

test_df = pd.read_csv(PATH + r"/test.csv")
train_df = pd.read_csv(PATH + r"/train.csv")

marged_df = pd.concat([test_df,train_df],axis=0)

In [4]:
#名称をカテゴリ変数に変換
cols_to_lower = ['brand', 'model', 'engine', 'fuel_type', 'transmission']

for col in cols_to_lower:
    marged_df[col] = marged_df[col].str.lower().str.strip()

In [5]:
#在庫数がわずかなブランドの削除
brands_to_exclude = ['smart', 'polestar']
rows_to_drop = marged_df[marged_df['brand'].isin(brands_to_exclude)].index
marged_df = marged_df.drop(rows_to_drop)

In [8]:
#エンジンの情報に燃料情報が含まれていれば燃料の欠損値補完
import re

fuel_pattern = r'(gasoline|diesel|e85 flex fuel|flex fuel|electric|hybrid|plug-in hybrid)'

for df in [marged_df]:
    df['fuel_type'] = df['fuel_type'].replace(['–', 'not supported'], np.nan)
    extracted = df['engine'].str.extract(fuel_pattern, expand=False, flags=re.IGNORECASE)
    df['fuel_type'] = df['fuel_type'].fillna(extracted)

for df in [marged_df]:
    df['fuel_type'] = df['fuel_type'].replace('e85 flex fuel', 'flex fuel')

In [20]:
#clean_titleをカテゴリ変数に変換
marged_df['clean_title'] = (marged_df['clean_title'] == 'Yes').astype(int)

#製造年数から経年数を算出
marged_df['age'] = 2026 - marged_df['model_year']

In [12]:
#エンジンの情報から馬力・排気量・気筒数を特徴量として作成
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

for df in [marged_df]:
    df['hp'] = df['engine'].str.extract(r'(\d+\.?\d*)hp', flags=re.IGNORECASE).astype(float)
    df['liters'] = df['engine'].str.extract(r'(\d+\.?\d*)l', flags=re.IGNORECASE).astype(float)
    df['cylinder'] = df['engine'].str.extract(r'(\d+\.?\d*) cylinder', flags=re.IGNORECASE).astype(float)

cols_to_impute = ['hp', 'liters', 'cylinder', 'milage', 'age']

ii_imputer = IterativeImputer(
    max_iter=10,
    random_state=42,
    initial_strategy='median'
)

marged_df[cols_to_impute] = ii_imputer.fit_transform(marged_df[cols_to_impute])

In [13]:
X, y = marged_df.drop('price', axis=1), marged_df['price']

cat_cols = X.select_dtypes(include=['object']).columns
num_cols = X.select_dtypes(exclude=['object']).columns

median = X[num_cols].median()

X[num_cols] = X[num_cols].fillna(median)
marged_df[num_cols] = marged_df[num_cols].fillna(median)

In [14]:
import pandas as pd
import numpy as np

# -----------------------------
# 1. 小文字化
# -----------------------------
marged_df["model"] = marged_df["model"].str.lower()
marged_df["brand"] = marged_df["brand"].str.lower()

# -----------------------------
# 2. 1～7単語目を分割
# -----------------------------
words = marged_df["model"].str.split(expand=True)

for i in range(7):
    if i in words.columns:
        marged_df[f"word{i+1}"] = words[i]
    else:
        marged_df[f"word{i+1}"] = np.nan

In [15]:
import pandas as pd


def extract_pure_model(row):
    # brandを小文字に統一（比較のブレを防ぐ）
    brand = (
        str(row["brand"]).lower().strip() if pd.notna(row["brand"]) else ""
    )

    w1 = row["word1"]
    w2 = row["word2"]
    w3 = row["word3"]
    w4 = row["word4"]

    # 欠損対策＋小文字に統一
    w1 = "" if pd.isna(w1) else str(w1).lower().strip()
    w2 = "" if pd.isna(w2) else str(w2).lower().strip()
    w3 = "" if pd.isna(w3) else str(w3).lower().strip()
    w4 = "" if pd.isna(w4) else str(w4).lower().strip()

    # ─── ① Rover (Land Rover) ───
    if w1 == "rover":
        if w2 == "range":
            if w4 in ["velar", "evoque", "sport"]:
                return w2 + w3 + w4
            else:
                return w2 + w3
        else:
            return w2

    # ─── ② 元のコードで Rover ブロックの外にあった Range 判定（安全のため統合） ───
    # 1単語目が Rover ではなく、2単語目が Range から始まるイレギュラー対策
    if w2 == "range":
        if w4 in ["velar", "evoque", "sport"]:
            return w2 + w3 + w4
        else:
            return w2 + w3

    # ─── ③ Mercedes ───
    if w1 in ["c-class", "cls-class"]:
        return w2 + w3

    if w1 == "amg":
        return w2

    if w1 == "alpina":
        return w2

    # ─── ④ Audi RS ───
    if brand == "audi" and w1 == "rs":
        return w1 + w2

    # ─── ⑤ Grand Cherokee ───
    if w1 == "grand" and w2 == "cherokee":
        return w1 + w2

    # ─── ⑥ Continental GT ───
    if w1 == "continental" and w2 == "gt":
        return w1 + w2

    # ─── ⑦ Tesla Model（Audi model 仕様書反映） ───
    if (brand == "tesla" and w1 == "model") or (
        brand == "audi" and w1 == "model"
    ):
        return w1 + w2

    # ─── ⑧ 2単語目が無い ───
    if w2 == "":
        return w1

    # ─── ⑨ その他 ───
    return w1


# 🚀 実行コード
marged_df["pure_model"] = marged_df.apply(extract_pure_model, axis=1)

In [18]:
import pandas as pd

# ─── 1. price が入っている（欠損していない）行を Train にする ───
train_df = marged_df[marged_df["price"].notnull()].copy()

# ─── 2. price が欠損している行を Test にする ───
test_df = marged_df[marged_df["price"].isnull()].copy()

# ─── 3. Testデータから予測対象である 'price' 列を落とす（お作法） ───
test_df = test_df.drop(columns=["price"])

# 📊 分割後のデータ件数を確認
print(f"📁 元の全体のデータ数: {len(marged_df)}")
print(f"🚗 Train（学習用）のデータ数: {len(train_df)}")
print(f"🔮 Test（予測用）のデータ数 : {len(test_df)}")

📁 元の全体のデータ数: 314208
🚗 Train（学習用）のデータ数: 188523
🔮 Test（予測用）のデータ数 : 125685


In [ ]:
import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import root_mean_squared_error  # scikit-learn 1.4以降
from sklearn.model_selection import KFold

# ─── 1. 使用する特徴量（列）の定義 ───
features = [
    "brand",
    "model_since",
    "pure_model",
    "milage",  # 🔥 細かい数値も重要なので戻しました
    "fuel_type",
    "transmission",
    "accident",
    "clean_title",
    "displacement_l",
    "engine_type_clean",
    "milage_psychological_bin",  # あなたが考えた過学習防止カテゴリ
]

# ─── 2. 文字列データを「category型」に一括変換 ───
for col in features:
    if marged_df[col].dtype == "object":
        marged_df[col] = marged_df[col].astype("category")

# ─── 3. priceを基準にTrainとTestに切り分け ───
train_df = marged_df[marged_df["price"].notnull()].copy()
test_df = marged_df[marged_df["price"].isnull()].copy()

X = train_df[features]
y = train_df["price"]
# 💡 【超重要】ターゲットをここで対数変換する！
y_log = np.log1p(y)

X_test = test_df[features]

# ─── 4. 交差検証（5-Fold KFold）の準備 ───
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# スコアや予測結果を格納する箱（対数ではなく、すべて元のドル/円スケールで保存します）
oof_preds_original = np.zeros(len(X))
test_preds_original = np.zeros(len(X_test))
cv_scores_original = []

# ─── 5. 交差検証ループの実行 ───
print("🚀 LightGBM 5-Fold Cross Validation【対数変換版】開始...\n" + "─" * 45)

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y_log)):
    # データの分割（yは対数化した y_log を切り出す）
    X_train, y_train_log = X.iloc[train_idx], y_log.iloc[train_idx]
    X_val, y_val_log = X.iloc[val_idx], y_log.iloc[val_idx]

    # LightGBM専用のデータセット形式に変換
    train_dataset = lgb.Dataset(X_train, label=y_train_log)
    val_dataset = lgb.Dataset(X_val, label=y_val_log, reference=train_dataset)

    # ハイパーパラメータの設定
    params = {
        "objective": "regression",
        "metric": "rmse",  # 対数空間での誤差を最小化
        "learning_rate": 0.05,
        "max_depth": 6,  # 🌲 過学習を防ぐための制限
        "num_leaves": 31,
        "min_data_in_leaf": 50,
        "colsample_bytree": 0.7,
        "random_state": 42 + fold,
        "verbose": -1,
    }

    # モデルの訓練
    model = lgb.train(
        params,
        train_dataset,
        num_boost_round=2000,
        valid_sets=[train_dataset, val_dataset],
        callbacks=[
            lgb.early_stopping(stopping_rounds=50, verbose=False),
            lgb.log_evaluation(100),
        ],
    )

    # 1. 検証データでの予測（対数スケールで出てくる）
    val_preds_log = model.predict(X_val, num_iteration=model.best_iteration)

    # 2. 🔥 【重要】元のスケール（ドル/円）に逆変換！
    val_preds_original = np.expm1(val_preds_log)
    oof_preds_original[val_idx] = val_preds_original

    # 3. テストデータの予測（対数から元スケールに戻して平均化へ）
    test_preds_log = model.predict(X_test, num_iteration=model.best_iteration)
    test_preds_original += np.expm1(test_preds_log) / 5

    # 4. 元のスケールでの本当のRMSEスコアを計算
    y_val_original = np.expm1(y_val_log)
    fold_score = root_mean_squared_error(y_val_original, val_preds_original)
    cv_scores_original.append(fold_score)

    print(f"🔹 Fold {fold+1} 元スケール換算 RMSE: {fold_score:,.2f}")
    print("─" * 45)

# ─── 6. 全体の検証結果の表示 ───
oof_score = root_mean_squared_error(y, oof_preds_original)
print(f"🏆 5-Fold 平均 元スケール RMSE : {np.mean(cv_scores_original):,.2f}")
print(f"🏆 全体 OOF  元スケール RMSE  : {oof_score:,.2f}")

# test_dfに予測結果を格納
test_df["predicted_price"] = test_preds_original